# all-reduce-eval-metrics — ex2: sample-count-weighted eval mean via packed all_reduce

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `all-reduce-eval-metrics`. Running the final beacon cell reports progress against the `Distributed: all_reduce eval metrics` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: all_reduce eval metrics` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`all-reduce-eval-metrics`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "all-reduce-eval-metrics"
DD_SUBTOPIC = "Distributed: all_reduce eval metrics"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Sample-count-weighted eval mean (two all_reduces)

Ex1 averaged a SCALAR loss with `all_reduce(SUM) / world_size`. That assumes every rank evaluated the SAME number of samples. In real distributed eval, the last batch is short — rank 3 might see 13 samples while ranks 0–2 see 32 each. Naive `mean` over-weights rank 3.

Correct form: reduce `(sum_loss, count)` separately, then divide:

```python
stats = t.tensor([local_loss_sum, local_count], dtype=t.float32)
dist.all_reduce(stats, op=dist.ReduceOp.SUM)
global_mean = stats[0] / stats[1]
```

**Why one tensor, not two all_reduces.** Bandwidth — one network round-trip vs two. The two scalars get packed into a length-2 tensor and reduced together. Identical math result, half the latency.

**Trap.** `local_loss_sum` (NOT `local_mean`). If you reduce the per-rank MEAN you lose the count weight and we're back to ex1's bug. The numerator must be the unreduced sum.

### Exercise 2 — sample-count-weighted eval mean via packed all_reduce

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply a packed `(sum_loss, count)` `all_reduce(SUM)` followed by a single divide to compute the true global mean of an eval metric across ranks with unequal sample counts.
> Keywords: all_reduce, weighted-mean, uneven-batch, packed-tensor
> ```

**KCs targeted:** `pack-sum-and-count-into-one-tensor`, `weighted-mean-via-sum-divide`

Implement `ex2_weighted_eval_mean(rank, world_size, dist_module, local_loss_sum, local_count)`. The uneven-batch-correct version of ex1's eval mean:

1. Pack `(local_loss_sum, local_count)` into a single length-2 tensor: `stats = t.tensor([local_loss_sum, float(local_count)], dtype=t.float32)`. (Cast count to float — `all_reduce` requires a float tensor.)
2. Run ONE `all_reduce(SUM)` over the packed tensor. After this, `stats[0]` is the global sum of losses and `stats[1]` is the global sample count.
3. Compute the true weighted mean: `global_mean = stats[0] / stats[1]`. (Returned as a Python float via `.item()`.)
4. Return `global_mean`.

Why packed.** Two all_reduces double the network round-trip cost; one packed all_reduce halves it. Math is identical.

Why this matters (the bug ex1 hides).** If rank 0 sees 32 samples with mean loss 1.0 and rank 1 sees 8 samples with mean loss 5.0:
- Naive `mean of means` = (1 + 5) / 2 = 3.0 (WRONG — over-weights small rank).
- Weighted = (32*1 + 8*5) / (32 + 8) = 72/40 = 1.8 (RIGHT — every sample counted once).

Input: `rank`, `world_size` — ints; `dist_module` — torch.distributed or mock; `local_loss_sum` — float (sum, not mean!); `local_count` — int.
Output: `float` — true global mean, same on every rank.

In [ ]:
def ex2_weighted_eval_mean(rank: int, world_size: int, dist_module,
                           local_loss_sum: float, local_count: int) -> float:
    stats = t.tensor([local_loss_sum, float(local_count)], dtype=t.float32)
    dist_module.all_reduce(stats, op=dist_module.ReduceOp.SUM)
    global_mean = stats[0] / stats[1]
    return global_mean.item()


<details><summary>Solution</summary>

```python
def ex2_weighted_eval_mean(rank: int, world_size: int, dist_module,
                           local_loss_sum: float, local_count: int) -> float:
    stats = t.tensor([local_loss_sum, float(local_count)], dtype=t.float32)
    dist_module.all_reduce(stats, op=dist_module.ReduceOp.SUM)
    global_mean = stats[0] / stats[1]
    return global_mean.item()
```

**Why pack into one tensor.** Each `all_reduce` is a network round-trip; on a fast interconnect (NVLink, IB) each costs microseconds, but they add up across thousands of eval batches in a long training run. Packing two scalars into one tensor halves the all_reduce count.

**`local_loss_sum`, not `local_mean`.** This is the subtle bug fix vs ex1. If you reduce per-rank MEANS, the count weight is lost; the weighted-mean math no longer works. Pass the unreduced sum (or sum of loss × batch_size, depending on how you computed local loss).

**Float cast on count.** `all_reduce` requires a float tensor on gloo (and integer reduce is finicky on NCCL too). Cast `count` to `float32` on the way in; cast back if you really need an int (usually you don't — divide stays float).

**Division by zero edge case.** If EVERY rank has count 0 (rare), `stats[1] = 0` and division gives `inf`/`nan`. In practice the eval pipeline guards against this upstream; we trust the caller.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()